# RO change over time

## Imports

In [ ]:
import datetime
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import src.XRO
import copy
import scipy.stats
import warnings
import calendar

# import gsw

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## specify args

In [ ]:
## should we remove median?
REMOVE_MEDIAN = True

## specify variables
varnames = ["T_e", "T_w", "h_w_ohc"]
# varnames = ["T_e", "dT", "h_ohc"]

## Functions

In [ ]:
def get_rolling_std(data, n=20):
    """
    Get standard deviation, computing over time and ensemble member. To increase
    sample size for variance estimate, compute over time window of 2n+1
    years, centered at given year.
    """

    ## do the computation
    kwargs = dict(fn=np.std, n=n, reduce_ensemble_dim=False)
    data_std = src.utils.get_rolling_fn_bymonth(data, **kwargs)

    ## unstack year and month
    data_std = src.utils.unstack_month_and_year(data_std)

    return data_std


def get_fits_over_time(data_rolling, model, by_member=False, **fit_kwargs):
    """Get RO fits for each ensemble member as a function of time."""

    ## empty list to hold results
    fits = []

    ## loop through years
    for y in tqdm.tqdm(data_rolling.year):

        ## get data for year
        data_y = data_rolling.sel(year=y)

        if by_member:

            ## separate fit for each ensemble member
            fits_ = []
            for m in data_rolling.member:
                with warnings.catch_warnings(action="ignore"):
                    fits_.append(model.fit_matrix(data_y.sel(member=m), **fit_kwargs))
            fit = xr.concat(fits_, dim=data_y.member)

        else:

            ## fit for all ensemble members together
            with warnings.catch_warnings(action="ignore"):
                fit = model.fit_matrix(data_y, **fit_kwargs)

        ## track fits
        fits.append(fit.drop_vars(["time", "X", "Y", "Yfit"]))

    ## put back in xarray
    fits = xr.concat(fits, dim=data_rolling.year)

    return fits


def get_fits_over_time_wrapper(
    data_rolling, model, by_member=False, fname=None, **fit_kwargs
):
    """wrapper function to handle saving/loading"""

    ## function to compute fits
    get_fits = lambda: get_fits_over_time(
        data_rolling, model=model, by_member=by_member, **fit_kwargs
    )

    ## if fname not specified, compute without loading/saving
    if fname is None:
        fits = get_fits()

    else:

        ## full path to file
        fp = pathlib.Path(os.environ["SAVE_FP"], "fits_cesm", fname)

        ## load if it already exists
        if fp.is_file():
            fits = xr.open_dataset(fp)

        ## otherwise, compute and save
        else:
            fits = get_fits()
            fits.to_netcdf(fp)

    return fits


def get_sims_over_time(model, params, **simulation_kwargs):
    """Compute stats over time"""

    ## take ensemble mean if necessary
    if "member" in params.dims:
        params = params.mean("member")

    ## empty list to hold result
    sims = []

    ## loop through years
    for y in tqdm.tqdm(params.year):

        ## do simulation
        sim_y = model.simulate(fit_ds=params.sel(year=y), **simulation_kwargs)

        ## append
        sims.append(sim_y)

    ## put back in xarray
    sims = xr.concat(sims, dim=params.year)

    return sims


def save(fig, fname, dpi=800, do_save=False):
    """save figure to file"""

    ## get save directory
    save_dir = pathlib.Path(os.environ["SAVE_FP"], "pre-egu-updates")

    ## get fname
    fname = f"{fname}-{varnames[0]}-subtract_median_{REMOVE_MEDIAN}.pdf"

    if do_save:
        fig.savefig(save_dir / fname, dpi=dpi, format="pdf")

    return

## Load data

### T, h

In [ ]:
## open data
Th = src.utils.load_cesm_indices(load_z20=True, load_h_cust=True, max_grad=True)

#### Load ELI

In [ ]:
## Load ELI
eli_all = xr.open_dataset(pathlib.Path(DATA_FP, "cesm/eli.nc"))
eli_forced, eli = src.utils.separate_forced(eli_all)

## add to data
Th = xr.merge([Th, eli])

Scale it

#### Merged Niño index

In [ ]:
## function to get merged nino3/34 regions
def get_nino_merged(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(190, 270))
    return x.sel(idx).mean(["longitude", "latitude"])


def get_Te(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(230, 280))
    return x.sel(idx).mean(["longitude", "latitude"])

def get_Tw(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(180, 230))
    return x.sel(idx).mean(["longitude", "latitude"])


## spatial data
forced, anom = src.utils.load_consolidated()

## compute
Th["T_m"] = src.utils.reconstruct_wrapper(anom[["sst", "sst_comp"]], get_nino_merged)[
    "sst"
]
Th["T_e"] = src.utils.reconstruct_wrapper(anom[["sst", "sst_comp"]], get_Te)["sst"]
Th["T_w"] = src.utils.reconstruct_wrapper(anom[["sst", "sst_comp"]], get_Tw)["sst"]
Th["dT"] = Th["T_w"]-Th["T_e"]

#### OHC

In [ ]:
def load_consolidated_wide():
    """utility function to load consolidated data"""

    ## directory with data
    CONS_DIR = pathlib.Path(os.environ["DATA_FP"], "cesm", "consolidated_05")

    ## function to align and open
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    align_and_open = lambda fp: src.utils.align_pop_times(xr.open_dataset(fp), **kwargs)

    ## open data and align pop times
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    forced = align_and_open(CONS_DIR / "forced.nc")
    anom = align_and_open(CONS_DIR / "anom.nc")

    return forced, anom


forced_wide, anom_wide = load_consolidated_wide()

In [ ]:
lon_avg = lambda x, lons: x.sel(longitude=slice(*lons)).mean("longitude")

## compute ohc
Th["h_w_ohc"] = src.utils.reconstruct_wrapper(
    anom_wide[["T", "T_comp"]],
    lambda x: lon_avg(x.integrate("z_t"), (120, 180)),
)["T"]

## compute ohc
Th["h_ohc"] = src.utils.reconstruct_wrapper(
    anom_wide[["T", "T_comp"]],
    lambda x: lon_avg(x.integrate("z_t"), (120, 280)),
)["T"]


## compute ml temp
Th["T_3_ml"] = src.utils.reconstruct_wrapper(
    anom_wide[["T", "T_comp"]],
    lambda x: lon_avg(x.sel(z_t=slice(None, 50)).integrate("z_t"), (210, 270)),
)["T"]

### NHF

In [ ]:
## get NHF in Niño regions
nhf_3 = src.utils.reconstruct_wrapper(
    anom[["nhf", "nhf_comp"]], fn=src.utils.get_nino3
)["nhf"]
nhf_34 = src.utils.reconstruct_wrapper(
    anom[["nhf", "nhf_comp"]], fn=src.utils.get_nino34
)["nhf"]
nhf_m = src.utils.reconstruct_wrapper(anom[["nhf", "nhf_comp"]], fn=get_nino_merged)[
    "nhf"
]
T_m = src.utils.reconstruct_wrapper(anom[["sst", "sst_comp"]], fn=get_nino_merged)[
    "sst"
]

## convert to units of K/mo
sec_per_mo = 8.64e4 * 30
sec_per_yr = 12 * sec_per_mo
rho = 1.02e3
Cp = 4.2e3
H = 50

Q_34 = nhf_34 * sec_per_yr / (rho * Cp * H)
Q_3 = nhf_3 * sec_per_yr / (rho * Cp * H)
Q_m = nhf_m * sec_per_yr / (rho * Cp * H)

## merge with Th data
Th["Q_34"] = Q_34.sel(time=Th.time)
Th["Q_3"] = Q_3.sel(time=Th.time)
Th["Q_m"] = Q_m.sel(time=Th.time)

#### LOAD UPDATED ELI

In [ ]:
eli0 = xr.open_dataarray(DATA_FP / "cesm" / "eli_updated.nc")

Th = xr.merge([Th, eli0])

#### zonal gradient

In [ ]:
def get_ddx(x):
    """compute horizontal gradient"""

    ## ranges
    lat_range = dict(latitude=slice(-5, 5))
    lon_range_w = dict(longitude=slice(120, 160))
    lon_range_e = dict(longitude=slice(210, 270))

    ## spatial avg
    avg = lambda x: x.mean(["latitude", "longitude"])

    return avg(x.sel(**lon_range_w, **lat_range)) - avg(
        x.sel(**lon_range_e, **lat_range)
    )

In [ ]:
## compute dTdx
Th["dTdx_total"] = src.utils.reconstruct_wrapper(
    forced[["sst", "sst_comp"]],
    get_ddx,
)["sst"]

### preprocess

In [ ]:
## standardize (for convenience)
Th /= Th.std()

## get windowed data (used to estimate change in parameters over time)
Th_rolling = src.utils.get_windowed(Th, window_size=480, stride=120)

## drop last year because of NaNs
Th_rolling = Th_rolling.sel(year=slice(None, 2080))

## compute grad
Th_rolling["dTdx"] = Th_rolling["T_3"] - Th_rolling["T_4"]

#### eli

In [ ]:
# compute climatological gradient
dTdx_clim = Th_rolling["dTdx_total"].groupby("time.month").mean(["time"])

## get scaling factor
dTdx_scale = dTdx_clim / dTdx_clim.max()

## apply scaling
for v in list(Th_rolling):
    if "eli" in v:
        Th_rolling[f"{v}_scaled"] = Th_rolling[v].groupby("time.month") * dTdx_scale

        ## subtrack median
        grouped = Th_rolling[f"{v}_scaled"].groupby("time.month")
        Th_rolling[f"{v}_scaled_median"] = grouped - grouped.median(["member", "time"])

#### remove median

In [ ]:
## remove median (optional)
if REMOVE_MEDIAN:
    median = Th_rolling.groupby("time.month").median(["time", "member"])
    Th_rolling = Th_rolling.groupby("time.month") - median

## add constant
Th_rolling["ones"] = xr.ones_like(Th_rolling["T_3"])

## Compute time-varying RO parameters

### Fit model

Filenames:
- "T_3-h_w_ohc-zero_median_all-members.nc": ```maskb=["h"], maskNT=["T2", "T3", "TH"], maskNH=["T2"]```
- "T_3-h_w_ohc-zero_median.nc": same, but separate fits for each ensemble member
- "T_3-h_w_ohc-zero_median_simple.nc": ```maskNT=["T2", "T3"]```

In [ ]:
## parameters for fitting
MODEL = src.XRO.XRO(ncycle=12, ac_order=3, is_forward=True)
fit_kwargs = dict(ac_mask_idx=None, maskb=varnames[:2], maskc=varnames[:2])

## get fits
fits = get_fits_over_time_wrapper(
    Th_rolling[varnames],
    model=MODEL,
    by_member=False,
    fname=None,
    # fname=f"{varnames[0]}-{varnames[1]}-zero_median_simple.nc",
    **fit_kwargs,
)

## extract parameters
params = src.utils.get_params(fits=fits, model=MODEL)

## get change from initial period
delta_params = params - params.isel(year=0)

## Validate RO model

### Stochastic simulation

In [ ]:
# simulation specs
simulation_kwargs = dict(
    nyear=40,
    ncopy=1000,
    seed=1000,
    X0_ds=Th_rolling[varnames].isel(year=0, member=0, time=0),
    # X0_ds = None,
    noise_type="white",
    use_noise_cov=False,
    is_xi_stdac=False,
)

In [ ]:
## do simulations
sims = get_sims_over_time(model=MODEL, params=fits, **simulation_kwargs)

## resample to DJF
get_djf = lambda x: x.resample({"time": "QS-DEC"}).mean().isel(time=slice(4, -4, 4))

## resample to seasonal
sims_djf = get_djf(sims)
Th_djf = get_djf(Th_rolling[varnames])

### Compute stats

In [ ]:
def get_std(x):
    return x.std(["time", "member"])


def get_quantiles(x):
    return x.quantile(dim=["time", "member"], q=[0.05, 0.1, 0.5, 0.9, 0.95])


def get_stats(x):
    """compute relevant stats for given dataset"""

    ## empty dataset to hold results
    stats = xr.Dataset()

    ## helper func to convert to xr.DataArray
    to_da = lambda x: x.to_dataarray(dim="v")

    ## compute
    stats["sigma"] = to_da(get_std(x))
    stats["q"] = to_da(get_quantiles(x))

    return stats

In [ ]:
stats_ro = get_stats(sims_djf)
stats_gt = get_stats(Th_djf)

### Plot results

#### Quantiles

In [ ]:
for v in varnames:
    print(f"\n{v}")
    fig, axs = plt.subplots(1, 2, figsize=(6, 3), layout="constrained")
    for ax, stats_ in zip(axs, [stats_gt, stats_ro]):
        q = stats_["q"].sel(v=v)
    
        ax.plot(stats_.year, q.sel(quantile=0.95), c="r", label="El Niño")
        ax.plot(stats_.year, -q.sel(quantile=0.05), c="b", label="La Niña")
        ax.plot(
            stats_.year,
            0.5 * (q.sel(quantile=0.95) - q.sel(quantile=0.05)),
            label="Average",
            c="k",
        )
    
        ax_kwargs = dict(ls="--", c="gray", lw=0.8)
        src.utils.add_vticks(axs, xticks=[1870, 2010, 2050], xlines=[2010, 2050])
        # ax.set_ylim([0, None])
    
    src.utils.set_lims(axs)
    axs[1].set_yticks([])
    axs[1].legend()
    axs[0].set_title("CESM2")
    axs[1].set_title("RO")
    plt.show()

In [ ]:
# fig,ax = plt.subplots(figsize=(4,3))
# delta = lambda x : x-x.isel(year=0)
# for i in range(3):
#     ax.plot(fits.year, delta(fits["Lac"].isel(rankx=i, ranky=i).mean("cycle")))

# ax.axhline(0, ls="--", c="k")

In [ ]:
Th_ = Th_rolling.resample({"time": "QS-JAN"}).mean().isel(time=slice(4, -4, 4))

fig, axs = plt.subplots(1, 2, figsize=(8, 4), layout="constrained")
for ax, yr in zip(axs, [2010, 2080]):
    axs[0].scatter(
        Th_["T_e"].sel(year=yr),
        Th_["T_w"].sel(year=yr),
        s=3,
    )
    axs[1].scatter(
        Th_["T_e"].sel(year=yr),
        Th_["T_e"].sel(year=yr) - Th_["T_w"].sel(year=yr),
        s=3,
    )

src.utils.set_lims(axs)

#### PDFs

In [ ]:
def get_pdf(x, year, varname=varnames[0], edges=np.linspace(-4.5, 4.5, 20)):
    """compute pdf for given data"""

    ## reshape data
    x_ = x[varname].sel(year=year).stack(s=["time", "member"]).values

    ## compute
    pdf, _ = src.utils.get_empirical_pdf(x_, edges=edges)

    return pdf, edges

In [ ]:
## specify plot var
PLOT_VAR = varnames[1]

fig, axs = plt.subplots(1, 2, figsize=(6, 2.5), layout="constrained")

for ax, x, title in zip(axs, [Th_djf, sims_djf], ["CESM2", "RO"]):

    for y, kwargs in zip(
        [1870, 2010, 2080],
        [dict(color="gray", fill=True, alpha=0.2), dict(lw=2), dict(lw=2)],
    ):

        ## compute pdf for given year
        pdf_y, edges = get_pdf(x, year=y, varname=PLOT_VAR)

        ## plot
        ax.stairs(pdf_y, edges, label=y, **kwargs)

    ## label/format
    ax.set_title(title)
    ax.set_xlim([-4.3, 4.3])
    ax.set_xticks([-3, 0, 3])
    ax_kwargs = dict(ls="--", c="k", lw=0.8)
    ax.axvline(0, **ax_kwargs)
    ax.axvline(-1.5, **ax_kwargs)

## formatting
src.utils.set_lims(axs)
axs[1].set_yticks([])
axs[1].legend(prop=dict(size=10))
plt.show()

In [ ]:
# sel = lambda x : x.isel(cycle=slice(9,12)).mean("cycle")
sel = lambda x : x.mean("cycle")
fig,axs = plt.subplots(1,3,figsize=(9,2.5))

for i, l in enumerate([r"$T_e$", r"$T_w$"]):

    axs[0].plot(fits.year, sel(fits["Lac"].isel(ranky=i,rankx=i)), label=l)
    
    for ax, v in zip(axs[1:], ["NLb_Lac", "NLc_Lac"]):
        ax.plot(fits.year, sel(fits[v].isel(ranky=i)), label=l)
        
for ax in axs:
    ax.axhline(0, ls="--", c="k", lw=.8)
src.utils.add_vticks(axs, xticks=[2010], xlines=[2010])
axs[1].legend()
plt.show()

### Variance/skewness over time

In [ ]:
get_sigma = lambda x: x.std("time").quantile(dim="member", q=[0.1, 0.5, 0.9])
sigma_mod = get_sigma(Th_djf)
sigma_sim = get_sigma(sims_djf)

In [ ]:
## compute stats
get_stat = lambda x, fn: xr.apply_ufunc(
    fn, x, input_core_dims=[["time"]], kwargs={"axis": -1}
)
get_stat_wrapper = lambda x, fn: get_stat(x, fn).quantile(
    dim="member", q=[0.1, 0.5, 0.9]
)

# median_ot = get_stat(Th_djf, np.median)
skew_mod = get_stat_wrapper(Th_djf, scipy.stats.skew)
skew_sim = get_stat_wrapper(sims_djf, scipy.stats.skew)
# std_ot = get_stat(Th_djf, np.std)
# m3_ot = skew_ot * (std_ot**3)

In [ ]:
## specify colors
colors = sns.color_palette()

## plot setup
fig, axs = plt.subplots(1, 2, figsize=(6.5, 3), layout="constrained")

## plot data
for ax, stats in zip(axs, [[sigma_mod, sigma_sim], [skew_mod, skew_sim]]):
    for stat, c, ls, label in zip(stats, colors, ["-", "--"], ["CESM2", "RO"]):
        for q, lw in zip([0.5, 0.1, 0.9], [2, 1, 1]):
            label_ = label if (q == 0.5) else None
            ax.plot(
                stat.year,
                stat[PLOT_VAR].sel(quantile=q),
                c=c,
                lw=lw,
                ls=ls,
                label=label_,
            )

src.utils.add_vticks(axs, xticks=[1870, 2010, 2080], xlines=[2010])

axs[0].legend()
axs[1].axhline(0, ls="--", c="k", lw=0.8)
axs[0].set_ylabel("K")
axs[1].set_ylabel("non-dim")
axs[0].set_title(r"$\sigma(T)$ over time (DJF)")
axs[1].set_title(r"skew$(T)$ over time (DJF)")
plt.show()